# Laboratorio práctico de ETL — Kaismart Solutions S.A.S.

Maestría en Ciencia de Datos e IA · Universidad Autónoma de Occidente (UAO)

Este cuaderno desarrolla el proceso de extracción de las dos fuentes de información de la empresa:

| Fuente | Origen | Registros esperados |
|---|---|---|
| Sistema comercial | Base de datos MySQL `clientes`, tabla `ventas` | 5.000 |
| Sistema logístico | Archivo Excel `kaismart_eventos_logisticos.xlsx` | 50.000 |

**Alcance de esta entrega:** Parte 1 (extracción desde MySQL), Parte 2 (extracción desde Excel),
Parte 3 (comprensión inicial de los datasets) y Parte 4 (perfil inicial de calidad del dato).
En estas partes **no se convierten tipos, no se limpia y no se imputa nada**: los datos se guardan y se analizan tal como llegan.

## Arquitectura Medallion

| Capa | Contenido | Carpeta | Se usa en |
|---|---|---|---|
| **Bronze** | Datos originales, tal como se extrajeron de la fuente, sin transformar | `data/bronze/` | Partes 1 a 4 |
| **Silver** | Datos depurados: nulos tratados, duplicados eliminados, categorías estandarizadas, tipos corregidos | `data/silver/` | Parte 7 |
| **Gold** | Datos integrados (ventas + logística por `pedido_id`) listos para análisis | `data/gold/` | Parte 7 |

## Configuración

Se carga `config.yaml` (fuentes, rutas y autores) y el archivo `.env` (contraseña de la base de datos, para que no quede escrita en el código).
Las rutas son relativas a la raíz del proyecto.

In [1]:
from pathlib import Path
import os

import pandas as pd
import yaml
from dotenv import load_dotenv

# Raíz del proyecto: la carpeta desde la que se ejecuta el cuaderno
RAIZ = Path.cwd()
assert (RAIZ / "config.yaml").exists(), f"No se encontró config.yaml en {RAIZ}"

# Configuración general (config.yaml) y contraseña (.env)
with open(RAIZ / "config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)
load_dotenv(RAIZ / ".env")

# Rutas de las capas; se crean si no existen
rutas = {nombre: RAIZ / ruta for nombre, ruta in config["rutas"].items()}
for ruta in rutas.values():
    ruta.mkdir(parents=True, exist_ok=True)

print("Autores:", ", ".join(config["autores"]))
print("Raíz del proyecto:", RAIZ)
for nombre, ruta in rutas.items():
    print(f"  {nombre:<7}-> {ruta}")

Autores: Simón Colonia, Ingrid Valentina Sanchez Manzano
Raíz del proyecto: C:\Users\IngridValentinaSánch\repos\kaismart-etl
  raw    -> C:\Users\IngridValentinaSánch\repos\kaismart-etl\data\raw
  bronze -> C:\Users\IngridValentinaSánch\repos\kaismart-etl\data\bronze
  silver -> C:\Users\IngridValentinaSánch\repos\kaismart-etl\data\silver
  gold   -> C:\Users\IngridValentinaSánch\repos\kaismart-etl\data\gold


---
# Parte 1. Extracción desde la base de datos

Se sigue el orden pedido en el enunciado. Los pasos 2 a 5 y el 7 están dentro de un único bloque `try / except / finally`
para que **la conexión se cierre siempre**, incluso si la consulta falla.

## Paso 1. Importar las librerías necesarias

`mysql.connector` establece la conexión con MySQL y `Error` es la excepción base de esa librería.

In [2]:
import mysql.connector
from mysql.connector import Error

## Pasos 2 a 5 y 7. Conectar, consultar, extraer, guardar en `df_ventas` y cerrar

Dentro del bloque, en orden:

* **Paso 2 – Conexión:** con los datos de `config.yaml`, la contraseña de `.env` y `connection_timeout=15` (si el servidor no responde en 15 s, falla en lugar de quedarse esperando).
* **Paso 3 – Consulta:** primero un `SELECT COUNT(*)` para saber cuántos registros hay en el origen y luego `SELECT *` sobre la tabla `ventas`.
* **Paso 4 – Extracción:** `fetchall()` trae todos los registros disponibles.
* **Paso 5 – DataFrame:** se construye `df_ventas` con los registros y los nombres de columna del cursor.
  (No se usa `pd.read_sql` con la conexión directa porque pandas emite un warning: solo soporta oficialmente SQLAlchemy.)
* **Validación:** un `assert` comprueba que lo extraído coincide con lo que hay en el origen.
* **Paso 7 – Cierre:** en `finally` se cierran el cursor y la conexión.

In [3]:
cfg_mysql = config["fuentes"]["mysql"]
tabla = cfg_mysql["tabla"]

# Se inicializan en None para que el bloque finally pueda cerrar solo lo que llegó a abrirse
conn = None
cursor = None

try:
    # ---- Paso 2. Crear la conexión con la base de datos
    conn = mysql.connector.connect(
        host=cfg_mysql["host"],
        port=cfg_mysql["port"],
        user=cfg_mysql["user"],
        password=os.getenv("DB_PASSWORD"),
        database=cfg_mysql["database"],
        connection_timeout=15,
        use_pure=True,  # driver en Python puro: la extensión en C falla en algunos equipos Windows
    )
    cursor = conn.cursor()
    print(f"Conexión exitosa a la base de datos '{cfg_mysql['database']}' (MySQL {conn.server_info})")

    # ---- Paso 3. Consultar la tabla ventas
    cursor.execute(f"SELECT COUNT(*) FROM `{tabla}`")
    registros_origen = cursor.fetchone()[0]
    cursor.execute(f"SELECT * FROM `{tabla}`")

    # ---- Paso 4. Extraer todos los registros disponibles
    filas = cursor.fetchall()

    # ---- Paso 5. Almacenar el resultado en el DataFrame df_ventas
    df_ventas = pd.DataFrame(filas, columns=cursor.column_names)

    # ---- Validación de la extracción
    assert len(df_ventas) == registros_origen, (
        f"Se extrajeron {len(df_ventas)} registros pero el origen tiene {registros_origen}"
    )
    print(f"Registros en el origen (COUNT(*)):         {registros_origen:,}")
    print(f"Registros extraídos (df_ventas):           {len(df_ventas):,}")
    print(f"Registros esperados según el enunciado:    {cfg_mysql['registros_esperados']:,}")
    if len(df_ventas) != cfg_mysql["registros_esperados"]:
        print("ADVERTENCIA: la cantidad extraída no coincide con la esperada en el enunciado.")

except Error as e:
    print(f"Error de MySQL: {e}")
    raise

finally:
    # ---- Paso 7. Cerrar correctamente el cursor y la conexión
    if cursor is not None:
        cursor.close()
    if conn is not None and conn.is_connected():
        conn.close()
        print("Cursor y conexión con la base de datos cerrados correctamente.")

Conexión exitosa a la base de datos 'clientes' (MySQL 5.5.5-10.11.18-MariaDB-cll-lve)


Registros en el origen (COUNT(*)):         5,000
Registros extraídos (df_ventas):           5,000
Registros esperados según el enunciado:    5,000
Cursor y conexión con la base de datos cerrados correctamente.


## Paso 6. Comprobar que la extracción fue correcta

### 6.1 Dimensiones: `df_ventas.shape`

In [4]:
df_ventas.shape

(5000, 17)

### 6.2 Nombres de las columnas: `df_ventas.columns`

In [5]:
df_ventas.columns

Index(['id_venta', 'pedido_id', 'fecha_venta', 'id_cliente', 'ciudad', 'canal',
       'id_tienda', 'categoria', 'producto', 'cantidad', 'precio_unitario',
       'descuento_pct', 'valor_bruto', 'valor_descuento', 'valor_neto',
       'medio_pago', 'calificacion_cliente'],
      dtype='str')

### 6.3 Primeros cinco registros: `df_ventas.head()`

In [6]:
df_ventas.head()

,id_venta,pedido_id,fecha_venta,id_cliente,ciudad,canal,id_tienda,categoria,producto,cantidad,precio_unitario,descuento_pct,valor_bruto,valor_descuento,valor_neto,medio_pago,calificacion_cliente
0,1,PED000001,2026-01-18 22:06:15,CLI00011,Bogotá D.C.,Aplicación móvil,NaN,Deportes,Banda caminadora,1,2199900.00,10.00,2199900.00,219990.00,1979910.00,Tarjeta crédito,NaN
1,2,PED000002,2026-04-25 21:07:01,CLI00802,Cali,Página web,NaN,Deportes,Mancuernas ajustables,1,399900.00,10.00,399900.00,39990.00,359910.00,Tarjeta crédito,NaN
2,3,PED000003,2026-03-28 02:39:43,CLI00255,Medellín,Página web,NaN,Tecnología,Tablet 10 pulgadas,1,899900.00,10.00,899900.00,89990.00,809910.00,Tarjeta crédito,4.0
3,4,PED000004,2026-04-30 12:35:40,CLI00383,Medellín,Aplicación móvil,NaN,Electrodomésticos,Freidora de aire,2,472400.00,15.00,944800.00,141720.00,803080.00,PSE,5.0
4,5,PED000005,2026-04-04 00:18:12,CLI01286,Bogotá D.C.,Aplicación móvil,NaN,Deportes,Balón fútbol,1,119900.00,0.00,119900.00,0.00,119900.00,Tarjeta crédito,NaN


### 6.4 Muestra aleatoria de cinco registros: `df_ventas.sample(5, random_state=42)`

`sample(5)` elige cinco filas al azar. Con `random_state=42` se fija la semilla del generador aleatorio,
así que **la muestra es reproducible**: cada vez que se ejecute se obtienen las mismas cinco filas.

In [7]:
df_ventas.sample(5, random_state=42)

,id_venta,pedido_id,fecha_venta,id_cliente,ciudad,canal,id_tienda,categoria,producto,cantidad,precio_unitario,descuento_pct,valor_bruto,valor_descuento,valor_neto,medio_pago,calificacion_cliente
1501,1502,PED001502,2026-06-10 08:36:33,CLI02390,Bucaramanga,Tienda física,BGA-01,Tecnología,Monitor 24 pulgadas,2,713900.00,5.00,1427800.00,71390.00,1356410.00,Tarjeta débito,NaN
2586,2587,PED002587,2026-05-09 09:23:12,CLI01262,Bucaramanga,Aplicación móvil,NaN,Tecnología,Audífonos Bluetooth,2,149900.00,5.00,299800.00,14990.00,284810.00,Billetera digital,4.0
2653,2654,PED002654,2026-05-20 23:03:48,CLI01371,Cali,Aplicación móvil,NaN,Oficina,Base para portátil,1,112100.00,0.00,112100.00,0.00,112100.00,Tarjeta crédito,NaN
1055,1056,PED001056,2026-03-18 19:24:39,CLI00437,Medellín,Tienda física,MDE-01,Deportes,Bicicleta estática,2,1499900.00,0.00,2999800.00,0.00,2999800.00,Efectivo,5.0
705,706,PED000706,2026-06-06 19:01:06,CLI01135,Bogotá D.C.,Página web,NaN,Tecnología,Teclado mecánico,1,229900.00,5.00,229900.00,11495.00,218405.00,PSE,NaN


## Capa Bronze — ventas

Se guarda `df_ventas` en `data/bronze/ventas_bronze.csv` **exactamente como se extrajo** (sin transformar nada), sin índice y en UTF-8.

In [8]:
ruta_ventas_bronze = rutas["bronze"] / "ventas_bronze.csv"
df_ventas.to_csv(ruta_ventas_bronze, index=False, encoding="utf-8")
print(f"Bronze guardado: {ruta_ventas_bronze.relative_to(RAIZ)}  ({len(df_ventas):,} filas)")

Bronze guardado: data\bronze\ventas_bronze.csv  (5,000 filas)


---
# Parte 2. Extracción desde el archivo Excel

El archivo `kaismart_eventos_logisticos.xlsx` está en `data/raw/`. Se abre con `pd.ExcelFile` (motor `openpyxl`) dentro de un `with`,
de modo que el archivo se cierra automáticamente al terminar. Primero se listan las hojas y luego se lee la hoja configurada en `config.yaml`.
Si el archivo no existe, se lanza un `FileNotFoundError` con la ruta esperada.

In [9]:
cfg_excel = config["fuentes"]["excel"]
ruta_excel = rutas["raw"] / cfg_excel["archivo"]

if not ruta_excel.exists():
    raise FileNotFoundError(f"No se encontró el archivo Excel en la ruta esperada: {ruta_excel}")

with pd.ExcelFile(ruta_excel, engine="openpyxl") as xls:
    print("Hojas del archivo:", xls.sheet_names)
    df_logistica = pd.read_excel(xls, sheet_name=cfg_excel["hoja"])

print(f"Registros extraídos (df_logistica):        {len(df_logistica):,}")
print(f"Registros esperados según el enunciado:    {cfg_excel['registros_esperados']:,}")
if len(df_logistica) != cfg_excel["registros_esperados"]:
    print("ADVERTENCIA: la cantidad extraída no coincide con la esperada en el enunciado.")

Hojas del archivo: ['eventos_logisticos']


Registros extraídos (df_logistica):        50,000
Registros esperados según el enunciado:    50,000


## Comprobar que la extracción fue correcta

### Dimensiones: `df_logistica.shape`

In [10]:
df_logistica.shape

(50000, 14)

### Nombres de las columnas: `df_logistica.columns`

In [11]:
df_logistica.columns

Index(['evento_id', 'pedido_id', 'fecha_evento', 'estado_evento',
       'centro_logistico', 'ciudad_destino', 'transportadora', 'numero_guia',
       'fecha_prometida_entrega', 'tiempo_etapa_horas', 'costo_envio',
       'incidencia', 'observacion', 'Unnamed: 13'],
      dtype='str')

### Primeros cinco registros: `df_logistica.head()`

In [12]:
df_logistica.head()

,evento_id,pedido_id,fecha_evento,estado_evento,centro_logistico,ciudad_destino,transportadora,numero_guia,fecha_prometida_entrega,tiempo_etapa_horas,costo_envio,incidencia,observacion,Unnamed: 13
0,1,PED000001,2026-01-18 22:06:15,Pedido recibido,CEDI Bogotá,Bogotá D.C.,NaN,NaN,2026-01-20 14:06:15,NaN,8900,NaN,NaN,"=AI("""")"
1,2,PED000001,2026-01-18 22:36:53,Pago aprobado,CEDI Bogotá,Bogotá D.C.,NaN,NaN,2026-01-20 14:06:15,0.51,8900,NaN,NaN,NaN
2,3,PED000001,2026-01-18 23:31:22,Orden enviada a logística,CEDI Bogotá,Bogotá D.C.,NaN,NaN,2026-01-20 14:06:15,0.91,8900,NaN,NaN,NaN
3,4,PED000001,2026-01-19 02:03:55,En alistamiento,CEDI Bogotá,Bogotá D.C.,NaN,NaN,2026-01-20 14:06:15,2.54,8900,NaN,NaN,NaN
4,5,PED000001,2026-01-19 06:14:51,Empacado,CEDI Bogotá,Bogotá D.C.,NaN,NaN,2026-01-20 14:06:15,4.18,8900,NaN,NaN,NaN


### Muestra aleatoria de cinco registros: `df_logistica.sample(5, random_state=42)`

In [13]:
df_logistica.sample(5, random_state=42)

,evento_id,pedido_id,fecha_evento,estado_evento,centro_logistico,ciudad_destino,transportadora,numero_guia,fecha_prometida_entrega,tiempo_etapa_horas,costo_envio,incidencia,observacion,Unnamed: 13
33553,33554,PED003356,2026-04-04 09:51:57,En alistamiento,CEDI Pereira,Pereira,NaN,NaN,2026-04-06 02:08:22,1.78,9900,NaN,Pedido procesado dentro de la ventana prevista.,NaN
9427,9428,PED000943,2026-04-14 11:09:42,En tránsito,CEDI Bucaramanga,Bucaramanga,Servientrega,KMS77388424,2026-04-15 16:24:14,9.56,8900,NaN,NaN,NaN
199,200,PED000020,2026-02-08 21:23:51,Entregado,CEDI Bogotá,Bogotá D.C.,TCC,KMS46254024,2026-02-09 11:26:46,5.13,13900,NaN,NaN,NaN
12447,12448,PED001245,2026-05-02 00:24:54,En tránsito,CEDI Cali,Cali,Inter Rapidísimo,KMS96024511,2026-05-03 00:03:57,9.38,15900,NaN,NaN,NaN
39489,39490,PED003949,2026-06-03 14:44:47,Entregado,CEDI Bogotá,Bogotá D.C.,Envia,KMS90495560,2026-06-03 02:22:34,9.09,17490,NaN,NaN,NaN


## Capa Bronze — logística

Se guarda `df_logistica` en `data/bronze/logistica_bronze.csv` sin transformar, sin índice y en UTF-8.

In [14]:
ruta_logistica_bronze = rutas["bronze"] / "logistica_bronze.csv"
df_logistica.to_csv(ruta_logistica_bronze, index=False, encoding="utf-8")
print(f"Bronze guardado: {ruta_logistica_bronze.relative_to(RAIZ)}  ({len(df_logistica):,} filas)")

Bronze guardado: data\bronze\logistica_bronze.csv  (50,000 filas)


---
# Parte 3. Comprensión inicial de los datasets

En esta parte se explora cada DataFrame por separado para entender su estructura y contenido.
Los datos no se modifican: cuando es necesario convertir un tipo para medir algo (por ejemplo, texto a fecha), se hace sobre una copia.

Para clasificar las variables se asignó a cada una un **rol** según su significado, ya que el tipo que detecta pandas no siempre
lo refleja. Por ejemplo, `numero_guia` es texto pero funciona como identificador, y `calificacion_cliente` es numérica pero ordinal.

## 3.1 `df_ventas`

### Puntos 1 a 5. Registros, variables, tipos y no nulos

In [15]:
print("Número de registros:", df_ventas.shape[0])
print("Número de variables:", df_ventas.shape[1])
print("Nombres de las variables:", list(df_ventas.columns))

# Tipo de dato y cantidad de registros no nulos de cada variable
pd.DataFrame({
    "tipo": df_ventas.dtypes,
    "no_nulos": df_ventas.notna().sum(),
    "nulos": df_ventas.isna().sum(),
})

Número de registros: 5000
Número de variables: 17
Nombres de las variables: ['id_venta', 'pedido_id', 'fecha_venta', 'id_cliente', 'ciudad', 'canal', 'id_tienda', 'categoria', 'producto', 'cantidad', 'precio_unitario', 'descuento_pct', 'valor_bruto', 'valor_descuento', 'valor_neto', 'medio_pago', 'calificacion_cliente']


,tipo,no_nulos,nulos
id_venta,int64,5000,0
pedido_id,str,5000,0
fecha_venta,datetime64[us],5000,0
id_cliente,str,5000,0
ciudad,str,5000,0
canal,str,5000,0
id_tienda,str,1499,3501
categoria,str,5000,0
producto,str,5000,0
cantidad,int64,5000,0


In [16]:
df_ventas.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   id_venta              5000 non-null   int64         
 1   pedido_id             5000 non-null   str           
 2   fecha_venta           5000 non-null   datetime64[us]
 3   id_cliente            5000 non-null   str           
 4   ciudad                5000 non-null   str           
 5   canal                 5000 non-null   str           
 6   id_tienda             1499 non-null   str           
 7   categoria             5000 non-null   str           
 8   producto              5000 non-null   str           
 9   cantidad              5000 non-null   int64         
 10  precio_unitario       5000 non-null   object        
 11  descuento_pct         5000 non-null   object        
 12  valor_bruto           5000 non-null   object        
 13  valor_descuento       5000 no

### Puntos 6 a 9. Identificadores, categóricas, numéricas y fechas

In [17]:
# Clasificación hecha a mano según el significado de cada variable
identificadores_ventas = ["id_venta", "pedido_id", "id_cliente", "id_tienda"]
categoricas_ventas = ["ciudad", "canal", "categoria", "producto", "medio_pago"]
numericas_ventas = ["cantidad", "precio_unitario", "descuento_pct", "valor_bruto",
                    "valor_descuento", "valor_neto", "calificacion_cliente"]
fechas_ventas = ["fecha_venta"]

print("Identificadores:", identificadores_ventas)
print("Categóricas:", categoricas_ventas)
print("Numéricas:", numericas_ventas)
print("Fechas:", fechas_ventas)

Identificadores: ['id_venta', 'pedido_id', 'id_cliente', 'id_tienda']
Categóricas: ['ciudad', 'canal', 'categoria', 'producto', 'medio_pago']
Numéricas: ['cantidad', 'precio_unitario', 'descuento_pct', 'valor_bruto', 'valor_descuento', 'valor_neto', 'calificacion_cliente']
Fechas: ['fecha_venta']


### Rango de fechas

In [18]:
print("Primera venta:", df_ventas["fecha_venta"].min())
print("Última venta:", df_ventas["fecha_venta"].max())

# Cantidad de ventas por mes
df_ventas["fecha_venta"].dt.to_period("M").value_counts().sort_index()

Primera venta: 2026-01-01 09:08:59
Última venta: 2026-06-30 20:42:35


fecha_venta
2026-01    830
2026-02    764
2026-03    867
2026-04    851
2026-05    821
2026-06    867
Freq: M, Name: count, dtype: int64

### Punto 10. Otros aspectos de calidad

Se verifican algunas reglas que los datos deberían cumplir. Los montos se convierten a número en una copia, porque desde MySQL
llegan como `Decimal`.

In [19]:
# Copia de los montos convertidos a número (df_ventas no se modifica)
montos = df_ventas[["cantidad", "precio_unitario", "descuento_pct", "valor_bruto",
                    "valor_descuento", "valor_neto"]].astype(float)

bruto_calculado = (montos["cantidad"] * montos["precio_unitario"]).round(2)
descuento_calculado = (montos["valor_bruto"] * montos["descuento_pct"] / 100).round(2)
neto_calculado = (montos["valor_bruto"] - montos["valor_descuento"]).round(2)

print("Filas donde valor_bruto no coincide:", (montos["valor_bruto"] != bruto_calculado).sum())
print("Filas donde valor_descuento no coincide:", (montos["valor_descuento"] != descuento_calculado).sum())
print("Filas donde valor_neto no coincide:", (montos["valor_neto"] != neto_calculado).sum())
print("Filas con cantidad o valores en cero o negativos:", (montos[["cantidad", "precio_unitario", "valor_neto"]] <= 0).any(axis=1).sum())

Filas donde valor_bruto no coincide: 0
Filas donde valor_descuento no coincide: 0
Filas donde valor_neto no coincide: 0
Filas con cantidad o valores en cero o negativos: 0


In [20]:
# id_tienda según el canal
tienda_fisica = df_ventas["canal"] == "Tienda física"
print("Ventas en tienda física sin id_tienda:", df_ventas.loc[tienda_fisica, "id_tienda"].isna().sum())
print("Ventas de otros canales con id_tienda:", df_ventas.loc[~tienda_fisica, "id_tienda"].notna().sum())

# Categorías por producto
print("Máximo de categorías por producto:", df_ventas.groupby("producto")["categoria"].nunique().max())

# Clientes
compras_por_cliente = df_ventas["id_cliente"].value_counts()
print("Clientes distintos:", len(compras_por_cliente))
print("Clientes con más de una compra:", (compras_por_cliente > 1).sum())

Ventas en tienda física sin id_tienda: 0
Ventas de otros canales con id_tienda: 0
Máximo de categorías por producto: 1
Clientes distintos: 2292
Clientes con más de una compra: 1458


In [21]:
# Precios distintos por producto
print("Precios distintos por producto:", df_ventas.groupby("producto")["precio_unitario"].nunique().unique())

# ¿El precio cambia por mes, ciudad o canal? Si fuera así, dentro de cada grupo habría un solo precio
mes = df_ventas["fecha_venta"].dt.month
print("Máximo de precios por producto y mes:", df_ventas.groupby(["producto", mes])["precio_unitario"].nunique().max())
print("Máximo de precios por producto y ciudad:", df_ventas.groupby(["producto", "ciudad"])["precio_unitario"].nunique().max())
print("Máximo de precios por producto y canal:", df_ventas.groupby(["producto", "canal"])["precio_unitario"].nunique().max())

Precios distintos por producto: [5]


Máximo de precios por producto y mes: 5
Máximo de precios por producto y ciudad: 5


Máximo de precios por producto y canal: 5


### Resultados de `df_ventas`

**¿Qué representa una fila?** Una venta: un pedido de un cliente, de un solo producto, con su cantidad, ciudad, canal, montos,
medio de pago y calificación (si el cliente la dio). `id_venta` y `pedido_id` son únicos, por lo que cada venta corresponde a un pedido.

**Rango de fechas:** del 1 de enero al 30 de junio de 2026, con entre 764 y 867 ventas por mes.

| Aspecto | Resultado |
|---|---|
| Registros / variables | 5.000 / 17 |
| Identificadores | `id_venta`, `pedido_id`, `id_cliente`, `id_tienda` |
| Categóricas | `ciudad`, `canal`, `categoria`, `producto`, `medio_pago` |
| Numéricas | `cantidad`, `precio_unitario`, `descuento_pct`, `valor_bruto`, `valor_descuento`, `valor_neto`, `calificacion_cliente` |
| Fechas | `fecha_venta` |
| Variables con nulos | `id_tienda` (3.501) y `calificacion_cliente` (2.172) |

**Otros aspectos de calidad:**

1. Los montos son coherentes en todas las filas: el valor bruto, el descuento y el valor neto se calculan correctamente, y no hay valores en cero o negativos.
2. Los montos y `descuento_pct` están guardados como `object` (`Decimal`) y no como números.
3. `id_tienda` solo tiene valor en las ventas de tienda física.
4. Cada producto pertenece a una sola categoría.
5. Cada producto aparece con 5 precios distintos, y esa variación no se explica por el mes, la ciudad ni el canal. No hay un precio único por producto.
6. 2.292 clientes realizaron las 5.000 ventas; 1.458 de ellos compraron más de una vez.

## 3.2 `df_logistica`

### Puntos 1 a 5. Registros, variables, tipos y no nulos

In [22]:
print("Número de registros:", df_logistica.shape[0])
print("Número de variables:", df_logistica.shape[1])
print("Nombres de las variables:", list(df_logistica.columns))

# Tipo de dato y cantidad de registros no nulos de cada variable
pd.DataFrame({
    "tipo": df_logistica.dtypes,
    "no_nulos": df_logistica.notna().sum(),
    "nulos": df_logistica.isna().sum(),
})

Número de registros: 50000
Número de variables: 14
Nombres de las variables: ['evento_id', 'pedido_id', 'fecha_evento', 'estado_evento', 'centro_logistico', 'ciudad_destino', 'transportadora', 'numero_guia', 'fecha_prometida_entrega', 'tiempo_etapa_horas', 'costo_envio', 'incidencia', 'observacion', 'Unnamed: 13']


,tipo,no_nulos,nulos
evento_id,int64,50000,0
pedido_id,str,50000,0
fecha_evento,str,49935,65
estado_evento,str,50000,0
centro_logistico,str,49910,90
ciudad_destino,str,49955,45
transportadora,str,19912,30088
numero_guia,str,19932,30068
fecha_prometida_entrega,str,49945,55
tiempo_etapa_horas,float64,45000,5000


In [23]:
df_logistica.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   evento_id                50000 non-null  int64  
 1   pedido_id                50000 non-null  str    
 2   fecha_evento             49935 non-null  str    
 3   estado_evento            50000 non-null  str    
 4   centro_logistico         49910 non-null  str    
 5   ciudad_destino           49955 non-null  str    
 6   transportadora           19912 non-null  str    
 7   numero_guia              19932 non-null  str    
 8   fecha_prometida_entrega  49945 non-null  str    
 9   tiempo_etapa_horas       45000 non-null  float64
 10  costo_envio              50000 non-null  int64  
 11  incidencia               511 non-null    str    
 12  observacion              1723 non-null   str    
 13  Unnamed: 13              1 non-null      str    
dtypes: float64(1), int64(2), str(11)


### Puntos 6 a 9. Identificadores, categóricas, numéricas y fechas

In [24]:
# Clasificación hecha a mano según el significado de cada variable
identificadores_logistica = ["evento_id", "pedido_id", "numero_guia"]
categoricas_logistica = ["estado_evento", "centro_logistico", "ciudad_destino", "transportadora", "incidencia"]
numericas_logistica = ["tiempo_etapa_horas", "costo_envio"]
fechas_logistica = ["fecha_evento", "fecha_prometida_entrega"]
texto_libre_logistica = ["observacion"]

print("Identificadores:", identificadores_logistica)
print("Categóricas:", categoricas_logistica)
print("Numéricas:", numericas_logistica)
print("Fechas:", fechas_logistica)
print("Texto libre:", texto_libre_logistica)

Identificadores: ['evento_id', 'pedido_id', 'numero_guia']
Categóricas: ['estado_evento', 'centro_logistico', 'ciudad_destino', 'transportadora', 'incidencia']
Numéricas: ['tiempo_etapa_horas', 'costo_envio']
Fechas: ['fecha_evento', 'fecha_prometida_entrega']
Texto libre: ['observacion']


### Rango de fechas

Las fechas están guardadas como texto, así que se convierten en una copia para calcular el rango.

In [25]:
# Se convierten en una copia; errors="coerce" deja como nulo lo que no se pueda convertir
fecha_evento = pd.to_datetime(df_logistica["fecha_evento"], errors="coerce")
fecha_prometida = pd.to_datetime(df_logistica["fecha_prometida_entrega"], errors="coerce")

print("fecha_evento:", fecha_evento.min(), "a", fecha_evento.max())
print("fecha_prometida_entrega:", fecha_prometida.min(), "a", fecha_prometida.max())

# Si los nulos después de convertir son los mismos que antes, todos los textos eran fechas válidas
print("Textos que no se pudieron convertir en fecha_evento:", fecha_evento.isna().sum() - df_logistica["fecha_evento"].isna().sum())
print("Textos que no se pudieron convertir en fecha_prometida_entrega:", fecha_prometida.isna().sum() - df_logistica["fecha_prometida_entrega"].isna().sum())

fecha_evento: 2026-01-01 09:08:59 a 2026-07-03 10:34:14
fecha_prometida_entrega: 2026-01-03 01:45:01 a 2026-07-03 03:33:34


Textos que no se pudieron convertir en fecha_evento: 0
Textos que no se pudieron convertir en fecha_prometida_entrega: 0


### Punto 10. Otros aspectos de calidad

In [26]:
print("Valores de la columna 'Unnamed: 13':", df_logistica["Unnamed: 13"].dropna().tolist())

# Número de eventos por pedido
eventos_por_pedido = df_logistica["pedido_id"].value_counts()
print("\nPedidos según su número de eventos:")
print(eventos_por_pedido.value_counts().sort_index())

# Pedidos con más de un costo de envío
print("\nPedidos con más de un valor de costo_envio:", (df_logistica.groupby("pedido_id")["costo_envio"].nunique() > 1).sum())

df_logistica["estado_evento"].value_counts()

Valores de la columna 'Unnamed: 13': ['=AI("")']

Pedidos según su número de eventos:
count
8        2
9       94
10    4807
11      96
12       1
Name: count, dtype: int64

Pedidos con más de un valor de costo_envio: 0


estado_evento
Listo para despacho          5006
Despachado                   5005
Orden enviada a logística    5004
En alistamiento              5001
Pedido recibido              5000
Pago aprobado                4999
Empacado                     4998
En tránsito                  4998
Entregado                    4998
En ruta de entrega           4991
Name: count, dtype: int64

### Resultados de `df_logistica`

**¿Qué representa una fila?** Un evento logístico: el paso de un pedido por una etapa del proceso, con su fecha, centro logístico,
ciudad de destino y, desde el despacho, la transportadora y el número de guía. El proceso tiene 10 etapas, desde *Pedido recibido*
hasta *Entregado*, y la mayoría de pedidos (4.807 de 5.000) tiene exactamente 10 eventos.

**Rango de fechas:** `fecha_evento` va del 1 de enero al 3 de julio de 2026 y `fecha_prometida_entrega` del 3 de enero al 3 de julio de 2026.

| Aspecto | Resultado |
|---|---|
| Registros / variables | 50.000 / 14 |
| Identificadores | `evento_id`, `pedido_id`, `numero_guia` |
| Categóricas | `estado_evento`, `centro_logistico`, `ciudad_destino`, `transportadora`, `incidencia` |
| Numéricas | `tiempo_etapa_horas`, `costo_envio` |
| Fechas | `fecha_evento`, `fecha_prometida_entrega` |
| Texto libre | `observacion` |
| Variables sin nulos | `evento_id`, `pedido_id`, `estado_evento`, `costo_envio` |

**Otros aspectos de calidad:**

1. Las dos fechas están guardadas como texto, aunque todas tienen el mismo formato y se pueden convertir a fecha.
2. La columna `Unnamed: 13` no es una variable: es una columna del Excel que solo tiene una fórmula (`=AI("")`) en una celda.
3. No todos los pedidos tienen 10 eventos: hay pedidos con 8, 9, 11 y 12, lo que indica eventos duplicados o faltantes (se revisa en 4.3).
4. `costo_envio` tiene el mismo valor en todos los eventos de un pedido, es decir, es un costo del pedido y no del evento.

---
# Parte 4. Perfil inicial de calidad del dato

Se mide la calidad de cada DataFrame sin eliminar, imputar ni corregir ningún valor.

## 4.1 Valores nulos

In [27]:
def tabla_nulos(df):
    # Cantidad y porcentaje de nulos por variable, de mayor a menor
    tabla = pd.DataFrame({
        "nulos": df.isna().sum(),
        "porcentaje": (df.isna().mean() * 100).round(2),
    })
    return tabla.sort_values("porcentaje", ascending=False)


tabla_nulos(df_ventas)

,nulos,porcentaje
id_tienda,3501,70.02
calificacion_cliente,2172,43.44
id_venta,0,0.00
id_cliente,0,0.00
pedido_id,0,0.00
ciudad,0,0.00
canal,0,0.00
categoria,0,0.00
fecha_venta,0,0.00
producto,0,0.00


In [28]:
tabla_nulos(df_logistica)

,nulos,porcentaje
Unnamed: 13,49999,100.00
incidencia,49489,98.98
observacion,48277,96.55
transportadora,30088,60.18
numero_guia,30068,60.14
tiempo_etapa_horas,5000,10.00
centro_logistico,90,0.18
fecha_evento,65,0.13
fecha_prometida_entrega,55,0.11
ciudad_destino,45,0.09


### ¿Qué nulos son normales y cuáles son un problema?

Se revisa si los nulos tienen una explicación dentro del proceso.

In [29]:
# df_ventas: nulos de id_tienda por canal
df_ventas[df_ventas["id_tienda"].isna()]["canal"].value_counts()

canal
Página web          2020
Aplicación móvil    1481
Name: count, dtype: int64

In [30]:
# df_logistica: transportadora y numero_guia antes y después del despacho
etapas_antes_despacho = ["Pedido recibido", "Pago aprobado", "Orden enviada a logística",
                         "En alistamiento", "Empacado", "Listo para despacho"]
antes = df_logistica[df_logistica["estado_evento"].isin(etapas_antes_despacho)]
despues = df_logistica[~df_logistica["estado_evento"].isin(etapas_antes_despacho)]

print("Eventos antes del despacho:", len(antes))
print("  sin transportadora:", antes["transportadora"].isna().sum())
print("  sin numero_guia:", antes["numero_guia"].isna().sum())
print("Eventos desde el despacho:", len(despues))
print("  sin transportadora:", despues["transportadora"].isna().sum())
print("  sin numero_guia:", despues["numero_guia"].isna().sum())

Eventos antes del despacho:

 30008
  sin transportadora: 30008
  sin numero_guia: 30008
Eventos desde el despacho: 19992
  sin transportadora: 80
  sin numero_guia: 60


In [31]:
# tiempo_etapa_horas: ¿en qué etapa están los nulos?
print("Etapa de los nulos de tiempo_etapa_horas:")
print(df_logistica[df_logistica["tiempo_etapa_horas"].isna()]["estado_evento"].value_counts())
print("Total de eventos 'Pedido recibido':", (df_logistica["estado_evento"] == "Pedido recibido").sum())

# incidencia y observacion
print("\nEventos con incidencia:", df_logistica["incidencia"].notna().sum())
print("Eventos con incidencia y sin observación:",
      (df_logistica["incidencia"].notna() & df_logistica["observacion"].isna()).sum())

# ¿Estas variables tienen un solo valor por pedido?
for columna in ["centro_logistico", "ciudad_destino", "fecha_prometida_entrega"]:
    maximo = df_logistica.groupby("pedido_id")[columna].nunique().max()
    print(f"Máximo de valores distintos de {columna} en un mismo pedido: {maximo}")

Etapa de los nulos de tiempo_etapa_horas:
estado_evento
Pedido recibido    5000
Name: count, dtype: int64
Total de eventos 'Pedido recibido': 5000

Eventos con incidencia: 511
Eventos con incidencia y sin observación: 0
Máximo de valores distintos de centro_logistico en un mismo pedido: 1
Máximo de valores distintos de ciudad_destino en un mismo pedido: 1
Máximo de valores distintos de fecha_prometida_entrega en un mismo pedido: 1


### Clasificación de los nulos

**`df_ventas`**

| Variable | Nulos | % | Clasificación | Justificación |
|---|---|---|---|---|
| `id_tienda` | 3.501 | 70,02 | Normal | Solo aplica a ventas en tienda física; todos los nulos son ventas por app o web. |
| `calificacion_cliente` | 2.172 | 43,44 | Normal | El cliente decide si califica. Sin embargo, casi la mitad de las ventas no tiene calificación. |

**`df_logistica`**

| Variable | Nulos | % | Clasificación | Justificación |
|---|---|---|---|---|
| `Unnamed: 13` | 49.999 | 100,00 | Problema | No es una variable del proceso, es una columna sobrante del Excel. |
| `incidencia` | 49.489 | 98,98 | Normal | Solo se registra cuando hay una novedad (511 eventos). |
| `observacion` | 48.277 | 96,55 | Normal | Es un comentario opcional; todos los eventos con incidencia tienen observación. |
| `transportadora` | 30.088 | 60,18 | Normal, con 80 casos problema | Antes del despacho no hay transportadora. Pero en 80 eventos de despacho o posteriores falta, y ahí sí debería estar. |
| `numero_guia` | 30.068 | 60,14 | Normal, con 60 casos problema | La guía se asigna al despachar. Falta en 60 eventos de despacho o posteriores. |
| `tiempo_etapa_horas` | 5.000 | 10,00 | Normal | Todos los nulos son del evento *Pedido recibido*, que es la primera etapa y no tiene una anterior para medir el tiempo. |
| `centro_logistico` | 90 | 0,18 | Problema | Todo evento debería tener un centro logístico. |
| `fecha_evento` | 65 | 0,13 | Problema | Todo evento debería tener fecha. |
| `fecha_prometida_entrega` | 55 | 0,11 | Problema | Todo pedido debería tener una fecha prometida. |
| `ciudad_destino` | 45 | 0,09 | Problema | Todo pedido debería tener destino. |

Los nulos que representan un problema son pocos (menos del 0,2% en cada variable). En `centro_logistico`, `ciudad_destino` y
`fecha_prometida_entrega` el valor es el mismo en todos los eventos de un pedido, por lo que el dato faltante sí existe en otros eventos del mismo pedido.

## 4.2 Valores únicos

Se usa `nunique()` para contar los valores distintos de cada variable. Como criterio, una variable categórica tiene
**baja cardinalidad** si tiene 10 valores o menos, y **alta cardinalidad** si tiene más de 50.

In [32]:
# Valores distintos por variable, de mayor a menor
df_ventas.nunique().sort_values(ascending=False)

id_venta                5000
pedido_id               5000
fecha_venta             5000
id_cliente              2292
valor_neto              1325
valor_descuento          693
valor_bruto              448
precio_unitario          133
producto                  30
id_tienda                 10
medio_pago                 6
ciudad                     6
categoria                  5
calificacion_cliente       5
descuento_pct              5
cantidad                   4
canal                      3
dtype: int64

In [33]:
df_logistica.nunique().sort_values(ascending=False)

evento_id                  49900
fecha_evento               49765
pedido_id                   5000
numero_guia                 4999
fecha_prometida_entrega     4999
tiempo_etapa_horas          1748
costo_envio                   12
estado_evento                 10
observacion                    9
centro_logistico               6
ciudad_destino                 6
incidencia                     6
transportadora                 5
Unnamed: 13                    1
dtype: int64

### Identificadores: `id_venta`, `pedido_id` y `evento_id`

In [34]:
print("id_venta en df_ventas:", df_ventas["id_venta"].nunique(), "únicos de", len(df_ventas), "filas")
print("pedido_id en df_ventas:", df_ventas["pedido_id"].nunique(), "únicos de", len(df_ventas), "filas")
print("pedido_id en df_logistica:", df_logistica["pedido_id"].nunique(), "únicos de", len(df_logistica), "filas")
print("evento_id en df_logistica:", df_logistica["evento_id"].nunique(), "únicos de", len(df_logistica), "filas")

# Números entre 1 y 50.000 que no aparecen en evento_id
todos_los_numeros = set(range(1, 50001))
print("Números de evento_id que no aparecen:", len(todos_los_numeros - set(df_logistica["evento_id"])))

id_venta en df_ventas: 5000 únicos de 5000 filas
pedido_id en df_ventas: 5000 únicos de 5000 filas


pedido_id en df_logistica: 5000 únicos de 50000 filas
evento_id en df_logistica: 49900 únicos de 50000 filas
Números de evento_id que no aparecen: 100


In [35]:
# numero_guia: ¿una guía por pedido?
guias = df_logistica.dropna(subset=["numero_guia"])
pedidos_por_guia = guias.groupby("numero_guia")["pedido_id"].nunique()
print("Guías distintas:", guias["numero_guia"].nunique())
print("Guías que aparecen en más de un pedido:", (pedidos_por_guia > 1).sum())
pedidos_por_guia[pedidos_por_guia > 1]

Guías distintas: 4999
Guías que aparecen en más de un pedido: 1


numero_guia
KMS64861259    2
Name: pedido_id, dtype: int64

### Resultados de la cardinalidad

**Identificadores:**

* **`id_venta`**: 5.000 valores únicos en 5.000 filas. Identifica cada venta.
* **`pedido_id`**: en `df_ventas` también es único. En `df_logistica` tiene 5.000 valores en 50.000 filas porque cada pedido tiene varios eventos, así que ahí no identifica la fila.
* **`evento_id`**: debería ser único, pero tiene 49.900 valores únicos en 50.000 filas, es decir, **100 repetidos**. Además, de los números del 1 al 50.000 faltan 100.
* **`numero_guia`**: hay 4.999 guías para 5.000 pedidos, porque una misma guía aparece en dos pedidos distintos.
* `id_cliente` e `id_tienda` se repiten, lo cual es normal: un cliente puede comprar varias veces y una tienda hace muchas ventas.

**Alta cardinalidad:** ninguna variable categórica supera los 50 valores. La que más tiene es `producto`, con 30.

**Baja cardinalidad:** `canal` (3), `categoria` (5), `transportadora` (5), `ciudad` (6), `medio_pago` (6), `centro_logistico` (6),
`ciudad_destino` (6), `incidencia` (6) y `estado_evento` (10). Cada categoría está escrita de una sola forma.

**Numéricas con pocos valores:** `cantidad` (4), `descuento_pct` (5), `calificacion_cliente` (5) y `costo_envio` (12). También
`observacion`, aunque es texto libre, solo tiene 9 frases distintas.

## 4.3 Duplicados

In [36]:
print("Filas completamente duplicadas en df_ventas:", df_ventas.duplicated().sum())
print("Filas completamente duplicadas en df_logistica:", df_logistica.duplicated().sum())

print("\nid_venta repetidos:", df_ventas["id_venta"].duplicated().sum())
print("pedido_id repetidos en df_ventas:", df_ventas["pedido_id"].duplicated().sum())
print("evento_id repetidos:", df_logistica["evento_id"].duplicated().sum())

# Un pedido no debería pasar dos veces por la misma etapa
print("Pedido y etapa repetidos:", df_logistica.duplicated(subset=["pedido_id", "estado_evento"]).sum())

Filas completamente duplicadas en df_ventas: 0


Filas completamente duplicadas en df_logistica: 99



id_venta repetidos: 0
pedido_id repetidos en df_ventas: 0
evento_id repetidos: 100
Pedido y etapa repetidos: 100


In [37]:
# evento_id repetido que no es copia exacta: se quitan las copias exactas y se busca el id que sigue repetido
repetidos = df_logistica[df_logistica["evento_id"].duplicated(keep=False)]
sin_copias_exactas = repetidos.drop_duplicates()
sin_copias_exactas[sin_copias_exactas["evento_id"].duplicated(keep=False)]

,evento_id,pedido_id,fecha_evento,estado_evento,centro_logistico,ciudad_destino,transportadora,numero_guia,fecha_prometida_entrega,tiempo_etapa_horas,costo_envio,incidencia,observacion,Unnamed: 13
35781,35782,PED003579,2026-01-23 11:47:37,Pago aprobado,NaN,Pereira,NaN,NaN,2026-01-25 13:18:30,0.49,11900,NaN,NaN,NaN
45258,35782,PED003579,2026-01-23 11:47:37,Pago aprobado,CEDI Pereira,Pereira,NaN,NaN,2026-01-25 13:18:30,0.49,11900,NaN,NaN,NaN


### Pedidos con etapas faltantes

In [38]:
# Etapas distintas de cada pedido (el proceso tiene 10)
etapas_por_pedido = df_logistica.groupby("pedido_id")["estado_evento"].nunique()
print("Pedidos con menos de 10 etapas:", (etapas_por_pedido < 10).sum())

# Pedidos que no tienen cada etapa = 5.000 pedidos menos los que sí la tienen
pedidos_con_etapa = df_logistica.drop_duplicates(["pedido_id", "estado_evento"])["estado_evento"].value_counts()
faltantes = 5000 - pedidos_con_etapa
print("Total de etapas faltantes:", faltantes.sum())
faltantes.sort_values(ascending=False)

Pedidos con menos de 10 etapas: 98


Total de etapas faltantes: 100


estado_evento
En ruta de entrega           18
Orden enviada a logística    12
En tránsito                  12
Empacado                     10
En alistamiento              10
Pago aprobado                10
Entregado                    10
Pedido recibido               9
Listo para despacho           5
Despachado                    4
Name: count, dtype: int64

### Resultados de los duplicados

| Revisión | `df_ventas` | `df_logistica` |
|---|---|---|
| Filas completamente duplicadas | 0 | 99 |
| Identificador repetido | 0 (`id_venta` y `pedido_id`) | 100 (`evento_id`) |
| Mismo pedido en la misma etapa más de una vez | — | 100 |

* `df_ventas` no tiene duplicados.
* En `df_logistica` hay 100 eventos registrados dos veces: 99 son copias exactas y 1 (`evento_id` 35782) solo se diferencia en que una de las copias no tiene `centro_logistico`.
* Al mismo tiempo, 98 pedidos tienen etapas faltantes (100 etapas en total) y faltan 100 números de `evento_id`. Como las cifras coinciden, es posible que los duplicados hayan reemplazado a eventos originales.

No se eliminó ningún duplicado.

## 4.4 Tipos de datos

Se compara el tipo que detecta pandas con el tipo que debería tener cada variable según su significado.

In [39]:
df_ventas.dtypes

id_venta                         int64
pedido_id                          str
fecha_venta             datetime64[us]
id_cliente                         str
ciudad                             str
canal                              str
id_tienda                          str
categoria                          str
producto                           str
cantidad                         int64
precio_unitario                 object
descuento_pct                   object
valor_bruto                     object
valor_descuento                 object
valor_neto                      object
medio_pago                         str
calificacion_cliente           float64
dtype: object

In [40]:
# Los montos aparecen como object: se revisa el tipo real de un valor
print("Tipo de un valor de valor_neto:", type(df_ventas["valor_neto"].iloc[0]))
print("Tipo de un valor de descuento_pct:", type(df_ventas["descuento_pct"].iloc[0]))

Tipo de un valor de valor_neto: <class 'decimal.Decimal'>
Tipo de un valor de descuento_pct: <class 'decimal.Decimal'>


In [41]:
df_logistica.dtypes

evento_id                    int64
pedido_id                      str
fecha_evento                   str
estado_evento                  str
centro_logistico               str
ciudad_destino                 str
transportadora                 str
numero_guia                    str
fecha_prometida_entrega        str
tiempo_etapa_horas         float64
costo_envio                  int64
incidencia                     str
observacion                    str
Unnamed: 13                    str
dtype: object

### Columnas que deben revisarse

| Tipo de variable | Columna | Tipo detectado | Observación |
|---|---|---|---|
| Monetarias | `precio_unitario`, `valor_bruto`, `valor_descuento`, `valor_neto` | `object` | Están como `Decimal` y no como números, por lo que no se pueden usar directamente en cálculos estadísticos. |
| Numéricas | `descuento_pct` | `object` | Mismo caso que los montos. |
| Numéricas | `calificacion_cliente` | `float64` | Es un entero de 1 a 5; aparece como decimal porque tiene nulos. |
| Fechas | `fecha_evento`, `fecha_prometida_entrega` | `str` | Están como texto, así que no permiten calcular tiempos ni comparar fechas. |
| Categóricas | `estado_evento` | `str` | Las etapas tienen un orden que el texto no conserva. |
| Otra | `Unnamed: 13` | `str` | No corresponde a ninguna variable. |

Los demás tipos son coherentes: los identificadores `id_venta` y `evento_id` son enteros y los que tienen prefijo (`PED`, `CLI`, `KMS`)
son texto; `cantidad` y `costo_envio` son enteros; `tiempo_etapa_horas` es decimal; `fecha_venta` ya es fecha, y las categóricas son texto.

# Parte 8. Automatización del pipeline (orquestador)

Hasta aquí el ETL se ejecuta a mano, celda por celda. En esta parte el pipeline se ejecuta solo, con la librería `schedule`.

La librería funciona en tres pasos:

1. **Se define la tarea** como una función: `ejecutar_pipeline()` llama los pasos en orden (extraer → Silver → Gold).
2. **Se programa cuándo correrla**, por ejemplo `schedule.every().day.at("06:00").do(ejecutar_pipeline)`. Esto solo registra la tarea en una agenda; todavía no ejecuta nada.
3. **Un ciclo revisa la agenda**: `schedule.run_pending()` ejecuta las tareas a las que ya les llegó la hora.

`schedule` no tiene un reloj propio: depende de que el ciclo siga corriendo. Si el programa se cierra, la agenda se pierde.

## Pasos del pipeline

Cada paso es una función. La extracción repite la lógica de las Partes 1 y 2 y guarda en Bronze. Los archivos se sobrescriben en cada ejecución, así que correr el pipeline varias veces no duplica datos. Silver y Gold quedan pendientes hasta incorporar la limpieza e integración de la Parte 7.

In [ ]:
from datetime import datetime
import time

import schedule


def extraer_ventas():
    """MySQL -> Bronze: extrae la tabla ventas y la guarda en ventas_bronze.csv."""
    cfg_mysql = config["fuentes"]["mysql"]
    conn = mysql.connector.connect(
        host=cfg_mysql["host"],
        port=cfg_mysql["port"],
        user=cfg_mysql["user"],
        password=os.getenv("DB_PASSWORD"),
        database=cfg_mysql["database"],
        connection_timeout=15,
        use_pure=True,
    )
    try:
        cursor = conn.cursor()
        cursor.execute(f"SELECT * FROM `{cfg_mysql['tabla']}`")
        df = pd.DataFrame(cursor.fetchall(), columns=cursor.column_names)
        cursor.close()
    finally:
        conn.close()

    df.to_csv(rutas["bronze"] / "ventas_bronze.csv", index=False, encoding="utf-8")
    print(f"  Bronze ventas:    {len(df):,} filas")
    return df


def extraer_logistica():
    """Excel -> Bronze: lee los eventos logísticos y los guarda en logistica_bronze.csv."""
    cfg_excel = config["fuentes"]["excel"]
    df = pd.read_excel(rutas["raw"] / cfg_excel["archivo"], sheet_name=cfg_excel["hoja"], engine="openpyxl")

    df.to_csv(rutas["bronze"] / "logistica_bronze.csv", index=False, encoding="utf-8")
    print(f"  Bronze logística: {len(df):,} filas")
    return df


def transformar_silver(df_ventas, df_logistica):
    """Bronze -> Silver. TODO: incorporar la limpieza de la Parte 7."""
    print("  Silver: pendiente (Parte 7)")
    return df_ventas, df_logistica


def integrar_gold(df_ventas, df_logistica):
    """Silver -> Gold. TODO: incorporar la integración por pedido_id de la Parte 7."""
    print("  Gold:   pendiente (Parte 7)")


def ejecutar_pipeline():
    """Ejecuta los pasos en orden; si uno falla, los siguientes no se ejecutan."""
    inicio = datetime.now()
    print(f"[{inicio:%Y-%m-%d %H:%M:%S}] Inicio del pipeline")
    try:
        ventas = extraer_ventas()
        logistica = extraer_logistica()
        ventas, logistica = transformar_silver(ventas, logistica)
        integrar_gold(ventas, logistica)
        print(f"[{datetime.now():%Y-%m-%d %H:%M:%S}] Pipeline terminado en {(datetime.now() - inicio).total_seconds():.1f} s\n")
    except Exception as e:
        # El error se informa pero no detiene el orquestador: se reintenta en la siguiente ejecución
        print(f"[{datetime.now():%Y-%m-%d %H:%M:%S}] ERROR, el pipeline se detuvo: {e}\n")

## Programación con `schedule`

En producción el orquestador correría en un script aparte, sin fin, por ejemplo todos los días a las 6 a. m.:

```python
schedule.every().day.at("06:00").do(ejecutar_pipeline)

while True:
    schedule.run_pending()
    time.sleep(60)
```

En el cuaderno ese ciclo nunca terminaría y dejaría la celda bloqueada. Por eso la demostración programa el pipeline cada minuto y se detiene después de dos ejecuciones.

In [ ]:
FRECUENCIA_MINUTOS = 1
EJECUCIONES_DEMO = 2

ejecuciones = 0


def tarea():
    global ejecuciones
    ejecutar_pipeline()
    ejecuciones += 1


schedule.clear()  # evita tareas repetidas si la celda se ejecuta otra vez
schedule.every(FRECUENCIA_MINUTOS).minutes.do(tarea)
print(f"Pipeline programado cada {FRECUENCIA_MINUTOS} minuto(s). Próxima ejecución: {schedule.next_run():%H:%M:%S}\n")

while ejecuciones < EJECUCIONES_DEMO:
    schedule.run_pending()
    time.sleep(1)

schedule.clear()
print("Demostración terminada.")